# CGR-MAT v1.4 — Ultrasound-Linkage Cohort Fixed

This version corrects the dataset interpretation: all 463 confirmed appendicitis patients have observed severity labels, but 18 lack a usable numeric `US_Number` and therefore cannot be linked to raw ultrasound images. The supervised multimodal cohort is 445 patients (116 complicated, 329 uncomplicated).

**Run:** choose `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Allow Google Drive access.


In [ ]:
!pip -q install catboost==1.2.8 openpyxl==3.1.5 requests


In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT = Path('/content/drive')
try:
    drive.mount(str(MOUNT), force_remount=False, timeout_ms=120000)
except Exception as first_error:
    print('First Drive mount attempt failed:', first_error)
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount(str(MOUNT), force_remount=True, timeout_ms=120000)
DRIVE_ROOT = MOUNT / 'MyDrive'
if not DRIVE_ROOT.exists():
    raise RuntimeError('Google Drive is not mounted. Enable pop-ups/cookies and rerun this cell.')
print('✓ Google Drive is ready:', DRIVE_ROOT)


In [ ]:
import hashlib
import pandas as pd
import requests
from IPython.display import display

CACHE = DRIVE_ROOT / 'MAT-Appendix' / 'data_cache' / 'regensburg'
CACHE.mkdir(parents=True, exist_ok=True)
XLSX = CACHE / 'app_data.xlsx'
URL = 'https://zenodo.org/records/7711412/files/app_data.xlsx?download=1'
EXPECTED_MD5 = 'd17a803f5e27532e518676a38f588b59'

def md5(path):
    h = hashlib.md5()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

if not XLSX.exists() or md5(XLSX) != EXPECTED_MD5:
    print('Downloading official app_data.xlsx for schema preflight...')
    r = requests.get(URL, timeout=180)
    r.raise_for_status()
    XLSX.write_bytes(r.content)
if md5(XLSX) != EXPECTED_MD5:
    raise RuntimeError('Official app_data.xlsx checksum mismatch.')
df = pd.read_excel(XLSX, engine='openpyxl')
norm = lambda s: s.fillna('').astype(str).str.strip().str.lower().str.replace('_', ' ', regex=False)
diagnosis = norm(df['Diagnosis'])
severity = norm(df['Severity'])
confirmed = diagnosis.eq('appendicitis') & severity.isin({'complicated','uncomplicated'})
audit = df.loc[confirmed, ['US_Number','Diagnosis','Severity']].copy()
audit['target'] = severity.loc[audit.index].map({'uncomplicated':0,'complicated':1}).astype(int)
audit['us_number_numeric'] = pd.to_numeric(audit['US_Number'], errors='coerce')
linked = audit.loc[audit['us_number_numeric'].notna()]
unlinked = audit.loc[audit['us_number_numeric'].isna()]
table = pd.DataFrame([
 {'group':'all_confirmed_appendicitis_with_observed_severity','patients':len(audit),'complicated':int(audit.target.sum()),'uncomplicated':int((audit.target==0).sum())},
 {'group':'multimodal_linked_confirmed_appendicitis','patients':len(linked),'complicated':int(linked.target.sum()),'uncomplicated':int((linked.target==0).sum())},
 {'group':'confirmed_appendicitis_without_us_number','patients':len(unlinked),'complicated':int(unlinked.target.sum()),'uncomplicated':int((unlinked.target==0).sum())},
])
print('SCHEMA PREFLIGHT — MUST PASS BEFORE TRAINING')
display(table)
expected = [(463,118,345),(445,116,329),(18,2,16)]
actual = [tuple(map(int, row)) for row in table[['patients','complicated','uncomplicated']].to_numpy()]
if actual != expected:
    raise RuntimeError(f'Schema preflight failed. Expected {expected}, found {actual}.')
print('✓ Schema preflight passed: 463 tabular-labeled, 445 ultrasound-linked, 18 unlinked.')


In [ ]:
import os
import torch

os.environ['CGR_MAT_RUN_MODE']='full'
os.environ['CGR_MAT_USE_DRIVE']='1'
os.environ['CGR_MAT_FORCE_RESTART']='0'
os.environ['CGR_MAT_PRETRAINED']='1'
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun all cells.')
print('Run mode:', os.environ['CGR_MAT_RUN_MODE'])
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = 'a072de95190d214177bbf3091cf98ab982e9ce5e'
LOADER_URL = (
    'https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/'
    f'{SOURCE_COMMIT}/src/cgr_mat/cgr_mat_verified_loader_v1_4.py'
)
print('Loading pinned CGR-MAT v1.4 loader...')
print('Source commit:', SOURCE_COMMIT)
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
print('Loader SHA256:', hashlib.sha256(loader_bytes).hexdigest())
loader = loader_bytes.decode('utf-8')
exec(compile(loader, LOADER_URL, 'exec'), globals(), globals())


## Required pipeline audit

Before fold training, the pipeline must display the same three rows as the schema preflight: 463 total labeled confirmed appendicitis cases, 445 ultrasound-linked cases, and 18 cases without usable `US_Number`.

Artifacts are saved under `MyDrive/MAT-Appendix/cgr_mat_runs/cgr_mat_v1_4_full_<hash>/`, including `prognostic_cohort_linkage_audit.csv` and `confirmed_appendicitis_without_us_number.csv`.
